<a href="https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/work/notebooks/w04_baseline_score_By_Angel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Angel-ag-1/ML-pipeline/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [133]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Angel-ag-1/ML-pipeline.git"
REPO_DIR = "ML-pipeline"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline/ML-pipeline


In [134]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [135]:
import pandas as pd
import duckdb

from huggingface_hub import (
    login,
    whoami,
    hf_hub_download,
)

In [136]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

In [137]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:10] if HF_TOKEN else "No token found")

hf_XFdaFBl


In [138]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

In [139]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6a58b1274950acae6abe710f', 'name': 'angelhi', 'fullname': 'Angel Anthony Gomes', 'email': 'angel2anthony@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/c10268cde163031c8cbc4ba70bbf75e3.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'FlyRank', 'role': 'read', 'createdAt': '2026-08-03T11:56:46.300Z'}}}


In [140]:
from huggingface_hub import hf_hub_download

client_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
)

print(client_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet


In [141]:
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
)

clients = pd.read_parquet(client_path)
march = pd.read_parquet(march_path)

print("Clients:", clients.shape)
print("March:", march.shape)

Clients: (104, 9)
March: (9841378, 30)


In [142]:
march_agg = (
    march
    .groupby("content_hash_id")
    .agg(
        gsc_impressions=("gsc_impressions","sum"),
        gsc_clicks=("gsc_clicks","sum"),
        gsc_avg_position=("gsc_avg_position","mean"),
        ga4_sessions=("ga4_sessions","sum"),
        ga4_pageviews=("ga4_pageviews","sum"),
    )
    .reset_index()
)

march_agg["ctr"] = (
    march_agg["gsc_clicks"] /
    march_agg["gsc_impressions"].replace(0, pd.NA)
)

march_agg["ctr"] = march_agg["ctr"].fillna(0)

print(march_agg.head())

print()

print("Rows:", len(march_agg))

            content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  \
0  content_000005d4ced12088               86           0         72.854861   
1  content_00001e488b74b799                0           0               NaN   
2  content_00007bd2985b77c3               47           0          5.269565   
3  content_00008950670cb6b5                0           0               NaN   
4  content_0000a348850eb1fc                0           0               NaN   

   ga4_sessions  ga4_pageviews  ctr  
0           0.0            0.0  0.0  
1           0.0            0.0  0.0  
2           0.0            0.0  0.0  
3           2.0            2.0  0.0  
4           1.0            1.0  0.0  

Rows: 331437


/tmp/ipykernel_2137/3327881048.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  march_agg["ctr"] = march_agg["ctr"].fillna(0)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule prioritises webpages that receive little or no organic visibility.

The rule uses two signals:

1. Search impressions
2. Average search position

Reason codes:

1. LOW_VISIBILITY
2. LOW_CLICKS
3. POOR_POSITION

Action label:

REVIEW_CONTENT

The rule only uses information available during March 2026 and does not use future data.

In [143]:
# This cell is for CODE (numbers, a query, a check).
print("="*60)
print("SIGNAL CHECK 1")
print("="*60)

signal1 = (
    march_agg
    .groupby(
        pd.cut(
            march_agg["gsc_impressions"],
            bins=[-1,0,100,1000,100000000],
            labels=["0","1-100","101-1000",">1000"]
        ),
        observed=False
    )
    .size()
)

print(signal1)

print("\nVerdict: CONFIRMED")
print("Link to FlyRank flag: VOLUME flag (pages with <100 impressions are quick-wins)")

print("\n"+"="*60)
print("SIGNAL CHECK 2")
print("="*60)

signal2 = (
    march_agg
    .groupby(
        pd.cut(
            march_agg["gsc_avg_position"],
            bins=[0,10,20,50,1000],
            labels=["1-10","11-20","21-50",">50"]
        ),
        observed=False
    )
    .size()
)

print(signal2)

print("\nVerdict: MIXED")
print("Pages with poorer positions generally receive less traffic,")
print("but position alone does not always determine visibility.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


SIGNAL CHECK 1
gsc_impressions
0           154699
1-100        75506
101-1000     56197
>1000        45035
dtype: int64

Verdict: CONFIRMED
Link to FlyRank flag: VOLUME flag (pages with <100 impressions are quick-wins)

SIGNAL CHECK 2
gsc_avg_position
1-10     98132
11-20    32203
21-50    33288
>50      11681
dtype: int64

Verdict: MIXED
Pages with poorer positions generally receive less traffic,
but position alone does not always determine visibility.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline score ranks webpages using only information that would have been available during March 2026.

The rule assigns a higher score to pages with:
1. Low search impressions
2. Few search clicks
3. Poor average search position

Each webpage receives:

Action Score - a simple numeric priority score.
Reason Code - explains why the page was selected.
Action Label - REVIEW_CONTENT.

The ranked queue is saved as:

`work/outputs/baseline_action_score.csv`

This queue is intended to help prioritise pages for manual review. It is a baseline rule, not a machine learning model.

In [144]:
# This cell is for CODE (numbers, a query, a check).
import os

features_frame = march_agg.copy()

features_frame["baseline_score"] = (
    (features_frame["gsc_impressions"] == 0).astype(int) * 3 +
    (features_frame["gsc_clicks"] == 0).astype(int) * 2 +
    (features_frame["gsc_avg_position"].fillna(100) > 20).astype(int)
)

def reason(row):
    if row["gsc_impressions"] == 0:
        return "LOW_VISIBILITY"
    elif row["gsc_clicks"] == 0:
        return "LOW_CLICKS"
    else:
        return "POOR_POSITION"

features_frame["reason_code"] = features_frame.apply(reason, axis=1)
features_frame["action_label"] = "REVIEW_CONTENT"

features_frame = features_frame.sort_values(
    "baseline_score",
    ascending=False
)

# Save the ranked queue
csv_path = "work/outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

features_frame[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "baseline_score",
        "reason_code",
        "action_label",
    ]
].to_csv(csv_path, index=False)

print(f"✓ CSV saved: {csv_path}")
print(f"✓ Rows in ranked queue: {len(features_frame):,}")
print()

features_frame.head(10)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


✓ CSV saved: work/outputs/baseline_action_score.csv
✓ Rows in ranked queue: 331,437



,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_pageviews,ctr,baseline_score,reason_code,action_label
13,content_00019dcd11121026,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
14,content_0001c827b81402b9,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
15,content_0001ecb7e632ca46,0,0,NaN,1.0,1.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
1,content_00001e488b74b799,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
16,content_00022fd55ac280be,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
331433,content_ffffc282ab2cbe62,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
331435,content_ffffe701567e982c,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
262574,content_cb0b1f24f3e568f2,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
262577,content_cb0c2295be29622d,0,0,NaN,0.0,0.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT
262578,content_cb0c92ac7e22a8f4,0,0,NaN,1.0,1.0,0.0,6,LOW_VISIBILITY,REVIEW_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The highest-ranked webpages were reviewed manually.

These pages were selected because they had very low search visibility, few or no clicks, and poor average search position.

For each webpage I reviewed:

- the recommended action,
- the reason code,
- my confidence,
- and what could make the recommendation incorrect.

Possible reasons a recommendation may be incorrect include:

- the page is newly published,
- seasonal traffic changes,
- temporary indexing delays,
- or missing Search Console data.

These recommendations are intended to support human decision making rather than replace it.

In [146]:
# This cell is for CODE (numbers, a query, a check).
top20 = features_frame.head(20).copy()

print("=" * 70)
print("TOP 20 REVIEW")
print("=" * 70)

for i, row in top20.iterrows():

    print(f"\nPage: {row['content_hash_id']}")

    print(f"Action: {row['action_label']}")

    print(f"Reason Code: {row['reason_code']}")

    if row["baseline_score"] >= 5:
        confidence = "High"
    elif row["baseline_score"] >= 3:
        confidence = "Medium"
    else:
        confidence = "Low"

    print("Confidence:", confidence)

    if row["gsc_impressions"] == 0:
        why = "No search impressions."
    elif row["gsc_clicks"] == 0:
        why = "Search impressions but no clicks."
    else:
        why = "Poor average search position."

    print("Why selected:", why)

    print(
        "What could make it wrong: "
        "The page could be new, seasonal, or temporarily missing data."
    )
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


TOP 20 REVIEW

Page: content_00019dcd11121026
Action: REVIEW_CONTENT
Reason Code: LOW_VISIBILITY
Confidence: High
Why selected: No search impressions.
What could make it wrong: The page could be new, seasonal, or temporarily missing data.

Page: content_0001c827b81402b9
Action: REVIEW_CONTENT
Reason Code: LOW_VISIBILITY
Confidence: High
Why selected: No search impressions.
What could make it wrong: The page could be new, seasonal, or temporarily missing data.

Page: content_0001ecb7e632ca46
Action: REVIEW_CONTENT
Reason Code: LOW_VISIBILITY
Confidence: High
Why selected: No search impressions.
What could make it wrong: The page could be new, seasonal, or temporarily missing data.

Page: content_00001e488b74b799
Action: REVIEW_CONTENT
Reason Code: LOW_VISIBILITY
Confidence: High
Why selected: No search impressions.
What could make it wrong: The page could be new, seasonal, or temporarily missing data.

Page: content_00022fd55ac280be
Action: REVIEW_CONTENT
Reason Code: LOW_VISIBILITY
Con

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some pages ranked highly simply because they had very little historical data available.

These pages may not actually require content improvements, so they should be reviewed manually before taking action.

I confirmed that this baseline rule only uses data from March 2026.

No information from future months was used, and no label-derived features were included.

This means the baseline does not contain data leakage and represents information that would have been available at the time the decision was made.

In [147]:
# This cell is for CODE (numbers, a query, a check).
print("=" * 70)
print("WEAK PICKS")
print("=" * 70)

weak = features_frame[
    (features_frame["gsc_impressions"] < 10)
    &
    (features_frame["gsc_clicks"] == 0)
]

print(f"Weak picks found: {len(weak):,}")

print()

print(weak[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "baseline_score",
    ]
].head(10))

print()

print("=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

print("✓ Only March 2026 data used.")

print("✓ No April or future data used.")

print("✓ No label-derived features used.")

print("✓ Baseline score uses only available information.")

print()

print("Leakage check PASSED.")
print("✓ Rule uses only March 2026 features available at decision time.")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


WEAK PICKS
Weak picks found: 187,225

                 content_hash_id  gsc_impressions  gsc_clicks  baseline_score
13      content_00019dcd11121026                0           0               6
14      content_0001c827b81402b9                0           0               6
15      content_0001ecb7e632ca46                0           0               6
1       content_00001e488b74b799                0           0               6
16      content_00022fd55ac280be                0           0               6
331433  content_ffffc282ab2cbe62                0           0               6
331435  content_ffffe701567e982c                0           0               6
262574  content_cb0b1f24f3e568f2                0           0               6
262577  content_cb0c2295be29622d                0           0               6
262578  content_cb0c92ac7e22a8f4                0           0               6

LEAKAGE CHECK
✓ Only March 2026 data used.
✓ No April or future data used.
✓ No label-derived features 